# Lab 06: NFL Game Prediction (XGBoost)

This lab teaches you how to build an NFL game prediction model using XGBoost. You'll load nflverse data, engineer team-level features, train classifiers and regressors, evaluate predictions, and interpret feature importances.

By the end you will:

- Load NFL schedule and player stats data via `quantitative_sports.data.nfl`
- Build team-level features (PPG, yards per play, turnover rate, QB rating)
- Compute rolling features (last 4 games, last 8 games, season-to-date)
- Construct the training set with home/away feature pairs
- Train an XGBoost classifier to predict win/loss
- Train XGBoost regressors to predict margin and total
- Evaluate on a held-out test set (accuracy, log loss, calibration)
- Interpret feature importances
- Save and load the model
- Compare model predictions to market spreads

---

## Prerequisites

- **Labs 01-05 completed** — you understand backtesting, odds, and metrics
- **`[notebook]` extra installed** — includes `xgboost`, `scikit-learn`, `matplotlib`
  ```bash
  uv pip install quantitative_sports[notebook]
  # or: pip install xgboost scikit-learn matplotlib
  ```

### What Is XGBoost?

**XGBoost** (eXtreme Gradient Boosting) is an ensemble learning method that builds sequential decision trees, each correcting the errors of the previous one. It's widely used in sports prediction because:

- Handles missing data well
- Captures non-linear relationships
- Resistant to overfitting with proper regularization
- Provides feature importance scores
- Fast training and inference

### Key XGBoost Concepts

| Concept | Description |
|---|---|
| **Boosting** | Sequential trees that correct prior errors |
| **Learning rate** | Step size for each tree (lower = more conservative) |
| **Max depth** | Maximum tree depth (controls complexity) |
| **Subsample** | Fraction of rows used per tree (prevents overfitting) |
| **Colsample_bytree** | Fraction of features used per tree |

---

## Section 1: Setup — Imports

We'll use XGBoost for modeling, scikit-learn for evaluation, and Quant-Sports's NFL data pipeline for data.

In [ ]:
# Cell 4: Core imports
import json
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# XGBoost and sklearn
try:
    import xgboost as xgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        accuracy_score,
        log_loss,
        brier_score_loss,
        classification_report,
        confusion_matrix,
    )
    from sklearn.calibration import calibration_curve
    XGBOOST_AVAILABLE = True
    print(f'XGBoost version: {xgb.__version__}')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available. Install with: pip install xgboost')
    print('Will use synthetic data and simplified models.')

# Quant-Sports NFL model
from quantitative_sports.models.predictive.nfl_game_model import (
    NFLGamePredictor,
    NFLGameFeatures,
    NFLGamePrediction,
    NFLModelConfig,
    _generate_synthetic_dataset,
    feature_columns,
)
from quantitative_sports.data.nfl import NFLDataPipeline, NFLDataConfig

print('Imports loaded successfully.')

---

## Section 2: Load NFL Data

We'll try to load real nflverse data via `NFLDataPipeline`. If the data isn't available (no internet or caching issues), we'll fall back to synthetic data — Quant-Sports's `NFLGamePredictor` includes a built-in synthetic data generator.

In [ ]:
# Cell 6: Load NFL schedule data
#
# NFLDataPipeline provides access to:
# - nflfastR player stats (via NFLfastRSource)
# - ESPN injury reports (via ESPNInjurySource)
# - Pinnacle odds (via PinnacleNFLOddsSource)

USE_REAL_DATA = False  # Will be set to True if real data loads successfully

try:
    pipeline = NFLDataPipeline()
    # Try loading 2023 season data
    df_2023 = pipeline.nflfastr.get_season(2023)
    if df_2023 is not None and not df_2023.empty:
        USE_REAL_DATA = True
        print(f'Loaded real nflverse data: {len(df_2023)} rows')
        print(f'Columns: {list(df_2023.columns[:10])}...')
    else:
        print('nflverse data not available. Using synthetic data.')
except Exception as e:
    print(f'Could not load nflverse data: {e}')
    print('Falling back to synthetic data.')

# Always generate synthetic data as backup
print('\nGenerating synthetic NFL dataset for model training...')
synthetic_df = _generate_synthetic_dataset(n_games=800, seed=42)
print(f'Synthetic dataset: {len(synthetic_df)} games')
print(f'Columns: {list(synthetic_df.columns)}')
print(f'\nSample rows:')
print(synthetic_df.head())

---

## Section 3: Build Team-Level Features

The `NFLGameFeatures` dataclass contains 14 features for each game, capturing offensive and defensive strength for both home and away teams.

### Feature Descriptions

| Feature | Description |
|---|---|
| `home_ppg_for` | Home team points per game (offense) |
| `home_ppg_against` | Home team points allowed per game (defense) |
| `home_yards_per_play` | Home team yards per play |
| `home_turnover_rate` | Home team turnovers per game |
| `home_qb_rating` | Home team QB passer rating proxy |
| `away_ppg_for` | Away team points per game (offense) |
| `away_ppg_against` | Away team points allowed per game (defense) |
| `away_yards_per_play` | Away team yards per play |
| `away_turnover_rate` | Away team turnovers per game |
| `away_qb_rating` | Away team QB passer rating proxy |
| `ppg_differential` | home_ppg_for - away_ppg_for |
| `defense_differential` | away_ppg_against - home_ppg_against |
| `home_advantage` | Home field advantage (default 2.5 points) |
| `rest_advantage` | Days of rest advantage (home - away) |

In [ ]:
# Cell 8: Explore the feature columns and synthetic data

fc = feature_columns()
print(f'Feature columns ({len(fc)} features):')
for i, col in enumerate(fc, 1):
    print(f'  {i:2d}. {col}')

print(f'\nSynthetic dataset statistics:')
print(synthetic_df[fc].describe().round(2))

---

## Section 4: Compute Rolling Features

In practice, we compute features using rolling windows (last 4 games, last 8 games, season-to-date) to capture recent form. The synthetic data already includes these aggregate features, but let's understand how rolling features work.

In [ ]:
# Cell 10: Demonstrate rolling feature computation
#
# Rolling features capture recent team performance:
# - Last 4 games: Short-term form
# - Last 8 games: Medium-term trend
# - Season-to-date: Overall performance

# Example: compute rolling PPG for a hypothetical team
np.random.seed(42)
team_scores = np.random.normal(22, 6, 17).clip(7, 45)  # 17 games
team_weeks = pd.DataFrame({
    'week': range(1, 18),
    'points_scored': team_scores,
})

# Rolling averages
team_weeks['ppg_last4'] = team_weeks['points_scored'].rolling(4, min_periods=1).mean()
team_weeks['ppg_last8'] = team_weeks['points_scored'].rolling(8, min_periods=1).mean()
team_weeks['ppg_season'] = team_weeks['points_scored'].expanding().mean()

print('Rolling PPG Features for a Hypothetical Team:')
print(f"{'Week':>4} {'Scored':>7} {'L4':>7} {'L8':>7} {'Season':>7}")
print(f"{'-'*4} {'-'*7} {'-'*7} {'-'*7} {'-'*7}")
for _, row in team_weeks.iterrows():
    print(f"{int(row['week']):>4} {row['points_scored']:>7.1f} {row['ppg_last4']:>7.1f} {row['ppg_last8']:>7.1f} {row['ppg_season']:>7.1f}")

# Plot rolling features
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(team_weeks['week'], team_weeks['points_scored'], 'o-', label='Weekly Points', alpha=0.5)
ax.plot(team_weeks['week'], team_weeks['ppg_last4'], 's-', label='Last 4 Games', linewidth=2)
ax.plot(team_weeks['week'], team_weeks['ppg_last8'], '^-', label='Last 8 Games', linewidth=2)
ax.plot(team_weeks['week'], team_weeks['ppg_season'], 'D-', label='Season Avg', linewidth=2, alpha=0.7)
ax.set_xlabel('Week', fontsize=12)
ax.set_ylabel('Points Per Game', fontsize=12)
ax.set_title('Rolling PPG Features', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Section 5: Build the Training Set

For each game, we join home team features and away team features into a single row. The targets are:
- `home_win` (binary: 1 if home team won, 0 if away)
- `spread` (regression: home_score - away_score)
- `total` (regression: home_score + away_score)

In [ ]:
# Cell 12: Examine the training data structure
#
# The synthetic dataset has all the features we need.
# Let's verify the structure and targets.

df = synthetic_df.copy()

print('Training dataset shape:', df.shape)
print(f'\nFeatures ({len(fc)} columns):')
print(df[fc].head())
print(f'\nTargets:')
print(f'  home_win distribution: {df["home_win"].value_counts().to_dict()}')
print(f'  spread mean: {df["spread"].mean():.2f}, std: {df["spread"].std():.2f}')
print(f'  total mean: {df["total"].mean():.2f}, std: {df["total"].std():.2f}')
print(f'\nCorrelation of key features with home_win:')
for col in ['ppg_differential', 'defense_differential', 'home_advantage']:
    if col in df.columns:
        corr = df[col].corr(df['home_win'])
        print(f'  {col}: {corr:.4f}')

---

## Section 6: Train/Validation/Test Split

For time-series data like NFL games, we split chronologically:
- **Train**: Earlier games (learn patterns)
- **Validation**: Middle games (tune hyperparameters)
- **Test**: Latest games (final evaluation)

Since our synthetic data doesn't have real dates, we'll split by index position.

In [ ]:
# Cell 14: Split data into train/val/test
#
# For synthetic data: 60% train, 20% val, 20% test
# For real data: you'd split by season (e.g., 2022=train, 2023=val, 2024=test)

n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

X_train = train_df[fc]
y_train_win = train_df['home_win']
y_train_spread = train_df['spread']
y_train_total = train_df['total']

X_val = val_df[fc]
y_val_win = val_df['home_win']
y_val_spread = val_df['spread']
y_val_total = val_df['total']

X_test = test_df[fc]
y_test_win = test_df['home_win']
y_test_spread = test_df['spread']
y_test_total = test_df['total']

print(f'Train: {len(train_df)} games ({train_df["home_win"].mean():.2%} home wins)')
print(f'Val:   {len(val_df)} games ({val_df["home_win"].mean():.2%} home wins)')
print(f'Test:  {len(test_df)} games ({test_df["home_win"].mean():.2%} home wins)')

---

## Section 7: Train XGBoost Classifier (Win/Loss)

The classifier predicts whether the home team wins (binary classification). We use `NFLGamePredictor` which wraps XGBoost internally.

In [ ]:
# Cell 16: Train the NFL Game Predictor
#
# NFLGamePredictor trains three models:
# 1. Classifier: predict home_win (binary)
# 2. Spread regressor: predict margin (home - away)
# 3. Total regressor: predict total points

if XGBOOST_AVAILABLE:
    # Use the full dataset for training (the predictor handles its own split)
    predictor = NFLGamePredictor()
    metrics = predictor.train(df, verbose=True)
    
    print('\nTraining complete!')
    print(f'  Holdout accuracy: {metrics.get("holdout_acc", 0):.4f}')
    print(f'  Holdout spread RMSE: {metrics.get("holdout_spread_rmse", 0):.2f}')
    print(f'  Holdout total RMSE: {metrics.get("holdout_total_rmse", 0):.2f}')
    print(f'\nFeature importances (top 5):')
    sorted_importances = sorted(predictor.feature_importances.items(), key=lambda x: x[1], reverse=True)
    for name, importance in sorted_importances[:5]:
        print(f'  {name}: {importance:.4f}')
else:
    print('XGBoost not available. Cannot train model.')
    print('Install with: pip install xgboost')
    predictor = None

---

## Section 8: Manual XGBoost Training (Under the Hood)

For deeper understanding, let's also train the models manually using scikit-learn's API with XGBoost. This shows exactly what `NFLGamePredictor` does internally.

In [ ]:
# Cell 18: Manual XGBoost training
#
# Train each model separately to understand the process.

if XGBOOST_AVAILABLE:
    # ---- Classifier (win probability) ----
    clf = xgb.XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
    )
    clf.fit(X_train, y_train_win, eval_set=[(X_val, y_val_win)], verbose=False)
    
    # ---- Spread regressor ----
    spread_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
    )
    spread_model.fit(X_train, y_train_spread, eval_set=[(X_val, y_val_spread)], verbose=False)
    
    # ---- Total regressor ----
    total_model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=400,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
    )
    total_model.fit(X_train, y_train_total, eval_set=[(X_val, y_val_total)], verbose=False)
    
    print('All three models trained manually.')
    print(f'  Classifier: {clf.n_estimators} trees, max_depth={clf.max_depth}')
    print(f'  Spread model: {spread_model.n_estimators} trees, max_depth={spread_model.max_depth}')
    print(f'  Total model: {total_model.n_estimators} trees, max_depth={total_model.max_depth}')
else:
    print('XGBoost not available. Skipping manual training.')

---

## Section 9: Evaluate on the Test Set

We evaluate our models on the held-out test set using classification metrics (accuracy, log loss) and regression metrics (RMSE).

In [ ]:
# Cell 20: Evaluate model performance on test set

if XGBOOST_AVAILABLE:
    # Predictions on test set
    win_probs = clf.predict_proba(X_test)[:, 1]  # Probability of home win
    win_preds = (win_probs >= 0.5).astype(int)
    spread_preds = spread_model.predict(X_test)
    total_preds = total_model.predict(X_test)
    
    # Classification metrics
    acc = accuracy_score(y_test_win, win_preds)
    ll = log_loss(y_test_win, win_probs)
    brier = brier_score_loss(y_test_win, win_probs)
    
    # Regression metrics
    spread_rmse = np.sqrt(np.mean((spread_preds - y_test_spread) ** 2))
    total_rmse = np.sqrt(np.mean((total_preds - y_test_total) ** 2))
    spread_mae = np.mean(np.abs(spread_preds - y_test_spread))
    total_mae = np.mean(np.abs(total_preds - y_test_total))
    
    print('Test Set Evaluation:')
    print(f'\n  Classification (Home Win):')
    print(f'    Accuracy:     {acc:.4f} ({acc*100:.1f}%)')
    print(f'    Log Loss:     {ll:.4f}')
    print(f'    Brier Score:  {brier:.4f}')
    print(f'\n  Regression (Spread):')
    print(f'    RMSE:         {spread_rmse:.2f} points')
    print(f'    MAE:          {spread_mae:.2f} points')
    print(f'\n  Regression (Total):')
    print(f'    RMSE:         {total_rmse:.2f} points')
    print(f'    MAE:          {total_mae:.2f} points')
    
    # Confusion matrix
    cm = confusion_matrix(y_test_win, win_preds)
    print(f'\n  Confusion Matrix:')
    print(f'    Predicted Away  Predicted Home')
    print(f'    Actual Away:    {cm[0][0]:>4}          {cm[0][1]:>4}')
    print(f'    Actual Home:    {cm[1][0]:>4}          {cm[1][1]:>4}')
else:
    print('XGBoost not available. Cannot evaluate.')

---

## Section 10: Feature Importances

Feature importance tells us which features the model relies on most. XGBoost provides several importance types; we'll use 'gain' (average improvement in loss from splits on that feature).

In [ ]:
# Cell 22: Feature importance analysis and visualization

if XGBOOST_AVAILABLE:
    # Get feature importances from the classifier
    importances = clf.feature_importances_
    feat_imp = pd.DataFrame({
        'feature': fc,
        'importance': importances,
    }).sort_values('importance', ascending=True)
    
    # Plot top features
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Classifier feature importance
    axes[0].barh(feat_imp['feature'], feat_imp['importance'], color='#2E86AB')
    axes[0].set_xlabel('Importance (gain)', fontsize=11)
    axes[0].set_title('Win Classifier Feature Importance', fontsize=13, fontweight='bold')
    
    # Spread model feature importance
    spread_imp = pd.DataFrame({
        'feature': fc,
        'importance': spread_model.feature_importances_,
    }).sort_values('importance', ascending=True)
    axes[1].barh(spread_imp['feature'], spread_imp['importance'], color='#E63946')
    axes[1].set_xlabel('Importance (gain)', fontsize=11)
    axes[1].set_title('Spread Regressor Feature Importance', fontsize=13, fontweight='bold')
    
    # Total model feature importance
    total_imp = pd.DataFrame({
        'feature': fc,
        'importance': total_model.feature_importances_,
    }).sort_values('importance', ascending=True)
    axes[2].barh(total_imp['feature'], total_imp['importance'], color='#2A9D8F')
    axes[2].set_xlabel('Importance (gain)', fontsize=11)
    axes[2].set_title('Total Regressor Feature Importance', fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print top 5 features for each model
    print('\nTop 5 Features by Model:')
    print(f'\n  Win Classifier:')
    for _, row in feat_imp.sort_values('importance', ascending=False).head(5).iterrows():
        print(f'    {row["feature"]:<25} {row["importance"]:.4f}')
    
    print(f'\n  Spread Regressor:')
    for _, row in spread_imp.sort_values('importance', ascending=False).head(5).iterrows():
        print(f'    {row["feature"]:<25} {row["importance"]:.4f}')
    
    print(f'\n  Total Regressor:')
    for _, row in total_imp.sort_values('importance', ascending=False).head(5).iterrows():
        print(f'    {row["feature"]:<25} {row["importance"]:.4f}')

---

## Section 11: Predict a Single Game

Now let's use the trained model to predict a specific game. We'll create an `NFLGameFeatures` instance and call `predict()`.

In [ ]:
# Cell 24: Predict a single game using NFLGamePredictor
#
# We create NFLGameFeatures for a hypothetical matchup
# and use the trained predictor to get predictions.

if predictor is not None and predictor.is_trained():
    # KC Chiefs (home) vs BAL Ravens (away)
    # KC: high-scoring offense, good QB
    # BAL: solid defense
    kc_vs_bal = NFLGameFeatures(
        home_team='KC',
        away_team='BAL',
        season=2024,
        week=10,
        home_ppg_for=26.5,      # KC scores ~26.5 ppg
        home_ppg_against=20.2,   # KC allows ~20.2 ppg
        home_yards_per_play=5.8, # KC YPP
        home_turnover_rate=1.1,  # KC TOs/game
        home_qb_rating=95.3,     # Mahomes proxy
        away_ppg_for=23.8,       # BAL scores ~23.8 ppg
        away_ppg_against=21.5,   # BAL allows ~21.5 ppg
        away_yards_per_play=5.4, # BAL YPP
        away_turnover_rate=1.3,  # BAL TOs/game
        away_qb_rating=88.7,     # Jackson proxy
        ppg_differential=2.7,    # KC scores more
        defense_differential=1.3, # BAL allows more
        home_advantage=2.5,      # Standard HFA
        rest_advantage=1.0,      # KC has 1 day rest advantage
    )
    
    prediction = predictor.predict(kc_vs_bal)
    
    print('Game Prediction: KC Chiefs vs BAL Ravens')
    print(f'  Home win probability: {prediction.home_win_prob:.4f} ({prediction.home_win_prob*100:.1f}%)')
    print(f'  Away win probability: {prediction.away_win_prob:.4f} ({prediction.away_win_prob*100:.1f}%)')
    print(f'  Projected spread:     {prediction.proj_spread:+.1f} (home favored)')
    print(f'  Projected total:      {prediction.proj_total:.1f} points')
    print(f'\n  Winner: {"KC Chiefs" if prediction.home_win_prob > 0.5 else "BAL Ravens"}')
else:
    print('Model not trained. Using manual predictions instead.')
    if XGBOOST_AVAILABLE:
        # Manual prediction
        features = np.array([
            26.5, 20.2, 5.8, 1.1, 95.3,  # home stats
            23.8, 21.5, 5.4, 1.3, 88.7,  # away stats
            2.7, 1.3, 2.5, 1.0             # derived features
        ]).reshape(1, -1)
        
        win_prob = float(clf.predict_proba(features)[0, 1])
        spread_pred = float(spread_model.predict(features)[0])
        total_pred = float(total_model.predict(features)[0])
        
        print('\nManual Prediction: KC Chiefs vs BAL Ravens')
        print(f'  Home win probability: {win_prob:.4f} ({win_prob*100:.1f}%)')
        print(f'  Projected spread:     {spread_pred:+.1f}')
        print(f'  Projected total:      {total_pred:.1f}')

---

## Section 12: Probability Calibration

A well-calibrated model predicts probabilities that match observed frequencies. If we predict 60% home win, then home teams should win about 60% of the time in those games. Let's check calibration.

In [ ]:
# Cell 26: Probability calibration plot
#
# A perfectly calibrated model has predicted probabilities that
# match the observed frequency of the event.

if XGBOOST_AVAILABLE:
    fraction_positives, mean_predicted = calibration_curve(
        y_test_win, win_probs, n_bins=10
    )
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Perfect calibration line
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfectly Calibrated')
    
    # Model calibration
    ax.plot(mean_predicted, fraction_positives, 'o-', linewidth=2,
            color='#2E86AB', markersize=8, label='XGBoost Classifier')
    
    # Histogram of predictions
    ax2 = ax.twinx()
    ax2.hist(win_probs, bins=20, alpha=0.3, color='gray', label='Prediction Distribution')
    ax2.set_ylabel('Count', fontsize=11)
    
    ax.set_xlabel('Mean Predicted Probability', fontsize=12)
    ax.set_ylabel('Fraction of Positives', fontsize=12)
    ax.set_title('Calibration Curve — XGBoost Win Probability', fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Brier score decomposition
    print(f'\nCalibration Summary:')
    print(f'  Brier Score: {brier:.4f}')
    print(f'  Log Loss: {ll:.4f}')
    print(f'  Mean predicted prob: {win_probs.mean():.4f}')
    print(f'  Actual home win rate: {y_test_win.mean():.4f}')
    print(f'  Calibration error: {abs(win_probs.mean() - y_test_win.mean()):.4f}')

---

## Section 13: Save and Load the Model

Quant-Sports's `NFLGamePredictor` has built-in save/load methods that persist the trained models to disk.

In [ ]:
# Cell 28: Save and load the model
#
# The save() method writes three model files and a metadata file.
# The load() classmethod restores everything.

if predictor is not None and predictor.is_trained():
    import tempfile
    
    # Save to a temporary directory
    model_dir = Path(tempfile.mkdtemp(prefix='nfl_model_'))
    predictor.save(model_dir)
    
    print(f'Model saved to: {model_dir}')
    print(f'Files:')
    for f in sorted(model_dir.iterdir()):
        size = f.stat().st_size
        print(f'  {f.name}: {size:,} bytes')
    
    # Load the model back
    loaded_predictor = NFLGamePredictor.load(model_dir)
    print(f'\nModel loaded successfully: {loaded_predictor.is_trained()}')
    
    # Verify predictions match
    original_pred = predictor.predict(kc_vs_bal)
    loaded_pred = loaded_predictor.predict(kc_vs_bal)
    
    print(f'\nVerification:')
    print(f'  Original win prob: {original_pred.home_win_prob:.6f}')
    print(f'  Loaded win prob:   {loaded_pred.home_win_prob:.6f}')
    print(f'  Match: {abs(original_pred.home_win_prob - loaded_pred.home_win_prob) < 1e-6}')
    
    # Clean up
    import shutil
    shutil.rmtree(model_dir, ignore_errors=True)
    print(f'\nCleaned up temporary model directory.')
else:
    print('Model not trained. Cannot save.')

---

## Section 14: Compare to Market Spread

If our model predicts a spread of -7 but the market has -3.5, there's a 3.5-point disagreement. This edge can be converted into a bet.

In [ ]:
# Cell 30: Compare model predictions to market spreads
#
# For each game in the test set, compare the model's projected
# spread to the ppg_differential (a proxy for the market line).

if XGBOOST_AVAILABLE:
    # Create a comparison DataFrame
    comparison_df = pd.DataFrame({
        'actual_spread': y_test_spread.values,
        'predicted_spread': spread_preds,
        'actual_total': y_test_total.values,
        'predicted_total': total_preds,
        'ppg_diff': X_test['ppg_differential'].values,
        'home_win': y_test_win.values,
        'home_win_prob': win_probs,
    })
    
    # Disagreement: model vs ppg differential (market proxy)
    comparison_df['spread_edge'] = comparison_df['predicted_spread'] - comparison_df['ppg_diff']
    comparison_df['spread_error'] = comparison_df['predicted_spread'] - comparison_df['actual_spread']
    
    print('Model vs Market (PPG Differential as proxy):')
    print(f"{'Metric':<25} {'Value':>10}")
    print(f"{'-'*25} {'-'*10}")
    print(f"{'Spread RMSE':<25} {np.sqrt(np.mean(comparison_df['spread_error']**2)):>10.2f}")
    print(f"{'Mean Spread Edge':<25} {comparison_df['spread_edge'].mean():>10.2f}")
    print(f"{'Spread Edge Std':<25} {comparison_df['spread_edge'].std():>10.2f}")
    print(f"{'Abs Spread Edge > 3 pts':<25} {(comparison_df['spread_edge'].abs() > 3).mean():>10.2%}")
    
    # How often does the model agree with the market?
    same_side = ((comparison_df['predicted_spread'] > 0) & (comparison_df['ppg_diff'] > 0)) | \
                ((comparison_df['predicted_spread'] < 0) & (comparison_df['ppg_diff'] < 0))
    agreement_rate = same_side.mean()
    print(f"\n{'Model-Market Agreement':<25} {agreement_rate:>10.2%}")
    print(f"  (Model and market pick the same side)")
    
    # When model and market disagree, who's right?
    disagree = ~same_side
    if disagree.sum() > 0:
        model_correct_disagree = (
            ((comparison_df.loc[disagree, 'predicted_spread'] > 0) & (comparison_df.loc[disagree, 'actual_spread'] > 0)) |
            ((comparison_df.loc[disagree, 'predicted_spread'] < 0) & (comparison_df.loc[disagree, 'actual_spread'] < 0))
        ).mean()
        print(f"  When they disagree, model is right: {model_correct_disagree:.2%}")

---

## Section 15: Using the `build_features_from_pipeline` Helper

When real nflverse data is available, you can use `build_features_from_pipeline()` to automatically extract team-level features from the raw data.

In [ ]:
# Cell 32: Demonstrate build_features_from_pipeline
#
# This function extracts features from nflfastR data using
# the NFLDataPipeline. It's used internally by NFLGamePredictor.

from quantitative_sports.models.predictive.nfl_game_model import build_features_from_pipeline

if USE_REAL_DATA:
    # Build features for a specific game using real data
    real_features = build_features_from_pipeline(
        pipeline,
        home_team='KC',
        away_team='BAL',
        season=2023,
        week=10,
    )
    print('Features built from real data:')
    for field, value in asdict(real_features).items():
        if isinstance(value, float):
            print(f'  {field}: {value:.2f}')
        else:
            print(f'  {field}: {value}')
else:
    print('Real data not available. Showing synthetic feature example:')
    example_features = NFLGameFeatures(
        home_team='KC',
        away_team='BAL',
        season=2024,
        week=10,
        home_ppg_for=26.5,
        home_ppg_against=20.2,
        home_yards_per_play=5.8,
        home_turnover_rate=1.1,
        home_qb_rating=95.3,
        away_ppg_for=23.8,
        away_ppg_against=21.5,
        away_yards_per_play=5.4,
        away_turnover_rate=1.3,
        away_qb_rating=88.7,
        ppg_differential=2.7,
        defense_differential=1.3,
        home_advantage=2.5,
        rest_advantage=1.0,
    )
    print('\nExample NFLGameFeatures:')
    for field, value in asdict(example_features).items():
        if isinstance(value, float):
            print(f'  {field}: {value:.2f}')
        else:
            print(f'  {field}: {value}')
    
    print(f'\nFeature vector shape: {example_features.feature_vector().shape}')
    print(f'Feature vector: {example_features.feature_vector()}')
    
    print(f'\nAs DataFrame:')
    print(example_features.to_dataframe())

In [ ]:
# Cell 33: Prediction distribution visualization

if XGBOOST_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Win probability distribution
    axes[0].hist(win_probs, bins=30, color='#2E86AB', alpha=0.7, edgecolor='black')
    axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='50% threshold')
    axes[0].axvline(x=win_probs.mean(), color='orange', linestyle='--', linewidth=2,
                    label=f'Mean: {win_probs.mean():.3f}')
    axes[0].set_xlabel('Predicted Home Win Probability', fontsize=11)
    axes[0].set_ylabel('Count', fontsize=11)
    axes[0].set_title('Win Probability Distribution', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=9)
    
    # Spread prediction distribution
    axes[1].hist(spread_preds, bins=30, color='#E63946', alpha=0.7, edgecolor='black')
    axes[1].axvline(x=0, color='black', linestyle='--', linewidth=2, label='Pick\'em')
    axes[1].axvline(x=y_test_spread.mean(), color='orange', linestyle='--', linewidth=2,
                    label=f'Mean: {y_test_spread.mean():.1f}')
    axes[1].set_xlabel('Predicted Spread (Home - Away)', fontsize=11)
    axes[1].set_ylabel('Count', fontsize=11)
    axes[1].set_title('Predicted Spread Distribution', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=9)
    
    # Total prediction distribution
    axes[2].hist(total_preds, bins=30, color='#2A9D8F', alpha=0.7, edgecolor='black')
    axes[2].axvline(x=y_test_total.mean(), color='orange', linestyle='--', linewidth=2,
                    label=f'Mean: {y_test_total.mean():.1f}')
    axes[2].set_xlabel('Predicted Total Points', fontsize=11)
    axes[2].set_ylabel('Count', fontsize=11)
    axes[2].set_title('Predicted Total Distribution', fontsize=13, fontweight='bold')
    axes[2].legend(fontsize=9)
    
    plt.tight_layout()
    plt.show()
else:
    print('XGBoost not available. Skipping visualization.')

---

## Exercises

Try these on your own:

1. **Add a weather feature** — Incorporate game-day weather (temperature, wind speed, precipitation) into the feature set. How does this affect prediction accuracy?

2. **Try a different model** — Replace XGBoost with a RandomForest or LogisticRegression. Compare accuracy, log loss, and calibration.

3. **Predict playoffs** — Filter the dataset to only playoff games (weeks 18+). Does the model perform differently in the postseason?

4. **Hyperparameter tuning** — Use `GridSearchCV` or `Optuna` to optimize the XGBoost hyperparameters. Can you beat the default accuracy?

5. **Ensemble models** — Combine the XGBoost predictions with a simple baseline (e.g., always pick home team) using weighted averaging. Does the ensemble outperform either model alone?

---

## Summary

In this lab you learned:

- How to load NFL data via `NFLDataPipeline` and `NFLfastRSource`
- How to build team-level features using `NFLGameFeatures` (14 features per game)
- How to compute rolling features (last 4, last 8, season-to-date)
- How to train XGBoost models for classification (win/loss) and regression (spread, total)
- How to evaluate models with accuracy, log loss, calibration curves, and RMSE
- How to interpret feature importances to understand what drives predictions
- How to predict a single game using `NFLGamePredictor.predict()`
- How to save and load models for reuse
- How to compare model predictions to market spreads to find edges

### Key API Reference

| Class/Function | Module | Purpose |
|---|---|---|
| `NFLGamePredictor` | `quantitative_sports.models.predictive.nfl_game_model` | Train and predict NFL games |
| `NFLGameFeatures` | `quantitative_sports.models.predictive.nfl_game_model` | Feature container |
| `NFLGamePrediction` | `quantitative_sports.models.predictive.nfl_game_model` | Prediction result |
| `NFLModelConfig` | `quantitative_sports.models.predictive.nfl_game_model` | Hyperparameter config |
| `NFLDataPipeline` | `quantitative_sports.data.nfl` | Unified NFL data access |
| `_generate_synthetic_dataset()` | `quantitative_sports.models.predictive.nfl_game_model` | Synthetic data generator |
| `feature_columns()` | `quantitative_sports.models.predictive.nfl_game_model` | Canonical feature list |
| `build_features_from_pipeline()` | `quantitative_sports.models.predictive.nfl_game_model` | Feature engineering helper |

### Key Concepts

| Concept | Description |
|---|---|
| **XGBoost** | Gradient-boosted decision trees for classification/regression |
| **Feature engineering** | Converting raw stats into predictive features |
| **Rolling features** | Last N games averages capturing recent form |
| **Calibration** | How well predicted probabilities match observed frequencies |
| **Feature importance** | Which features contribute most to predictions |
| **Model spread** | Predicted margin used to identify betting edges |

### Next Steps

Continue to **Lab 07** to learn how to combine predictions with odds for positive-EV betting.

---

*This lab used `NFLGamePredictor` from `quantitative_sports.models.predictive.nfl_game_model`. For production use, train on real nflverse data with proper cross-validation.*